# PySpark: реальний ETL-пайплайн

Коротка лаба з задачами, які робить Data Engineer щодня.

---
**Що робимо:** читаємо → чистимо → трансформуємо → пишемо Parquet  
**Дані:** згенеровані продажі (100К рядків)

In [ ]:
try:
    from pyspark.sql import SparkSession
    from pyspark.sql.functions import (
        col, to_date, year, month, dayofmonth, dayofweek,
        when, coalesce, lit, sum as spark_sum, avg, count,
        row_number, datediff, current_date, round as spark_round, isnan
    )
    from pyspark.sql.window import Window
    from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, DateType
    SPARK_OK = True
except ImportError:
    SPARK_OK = False
    print('PySpark not installed. Install: pip install pyspark')

In [ ]:
if SPARK_OK:
    spark = SparkSession.builder \
        .appName('DE_PySpark_Lab') \
        .master('local[*]') \
        .config('spark.sql.adaptive.enabled', 'true') \
        .getOrCreate()
    
    spark.sparkContext.setLogLevel('WARN')
    print(f'Spark {spark.version} ready')
    print(f'Parallelism: {spark.sparkContext.defaultParallelism}')

---
## 1. Створюємо реалістичні дані

Генеруємо 100К продажів з дефектами (null, дублікати, сміття).

In [ ]:
if SPARK_OK:
    import random
    import datetime
    random.seed(42)

    customers = [f'C{str(i).zfill(5)}' for i in range(1, 1001)]
    products = ['Laptop', 'Phone', 'Tablet', 'Monitor', 'Keyboard', 'Mouse', 'Headphones', 'Charger', 'Cable', 'Webcam']
    categories = ['Electronics', 'Accessories', 'Peripherals']
    payment = ['Card', 'Cash', 'PayPal', 'Bank Transfer']

    rows = []
    for i in range(100000):
        row = [
            i + 1,
            random.choice(customers),
            random.choice(products),
            random.choice(categories),
            round(random.uniform(10, 2000), 2),
            random.randint(1, 5),
            datetime.date(2023, 1, 1) + datetime.timedelta(days=random.randint(0, 730)),
            random.choice(payment),
            random.choice(['completed', 'pending', 'cancelled', 'refunded'])
        ]
        rows.append(row)

    schema = StructType([
        StructField('order_id', IntegerType(), True),
        StructField('customer_id', StringType(), True),
        StructField('product', StringType(), True),
        StructField('category', StringType(), True),
        StructField('price', DoubleType(), True),
        StructField('quantity', IntegerType(), True),
        StructField('order_date', DateType(), True),
        StructField('payment_method', StringType(), True),
        StructField('status', StringType(), True),
    ])

    df_raw = spark.createDataFrame(rows, schema)
    print(f'Created: {df_raw.count():,} rows, {len(df_raw.columns)} columns')
    df_raw.show(5, truncate=False)

In [ ]:
# Додаємо дефекти (як у реальному житті)
if SPARK_OK:
    import random as rnd
    rnd.seed(7)
    
    defects = df_raw.collect()
    for i in [0, 500, 3000, 15000, 50000]:  # null у price
        if i < len(defects):
            defects[i] = list(defects[i])
            defects[i][4] = None
    
    for i in [100, 2500, 20000]:  # null у status
        if i < len(defects):
            defects[i] = list(defects[i])
            defects[i][8] = None
    
    for i in [3, 4, 5]:  # дублікати Рядка 3
        if i < len(defects):
            defects[i] = list(defects[i])
            defects[i][1] = defects[3][1]
            defects[i][2] = defects[3][2]
            defects[i][3] = defects[3][3]
            defects[i][4] = defects[3][4]
            defects[i][5] = defects[3][5]
            defects[i][6] = defects[3][6]
            defects[i][7] = defects[3][7]
            defects[i][8] = defects[3][8]
    
    df = spark.createDataFrame(defects, schema)
    print('Дефекти додано: null price, null status, дублікати')
    print(f'Rows: {df.count():,}')

---
## 2. Data Quality — чистимо дані

Реальна задача DE: знайти і виправити проблеми.

In [ ]:
if SPARK_OK:
    print('=== Quality Report ===')
    
    total = df.count()
    
    # nulls
    for col_name in ['price', 'quantity', 'status', 'customer_id', 'order_date']:
        nulls = df.filter(col(col_name).isNull()).count()
        if nulls > 0:
            print(f'  {col_name}: {nulls} nulls ({nulls/total*100:.1f}%)')
    
    # duplicates
    dups = df.groupBy(df.columns).count().filter(col('count') > 1).count()
    print(f'  Дублікатів рядків: {dups}')
    
    # negative price
    neg = df.filter(col('price') < 0).count()
    print(f'  Negative price: {neg}')

In [ ]:
if SPARK_OK:
    # 1. Видаляємо дублікати
    df_dedup = df.dropDuplicates()
    print(f'Дублікатів видалено: {df.count() - df_dedup.count()}')

    # 2. Заповнюємо null у price медіаною (реальний кейс)
    median_price = df_dedup.approxQuantile('price', [0.5], 0.01)[0]
    df_clean = df_dedup.fillna({'price': median_price})
    
    # 3. null статус -> 'unknown'
    df_clean = df_clean.fillna({'status': 'unknown'})
    
    # 4. Додаємо total_amount
    df_clean = df_clean.withColumn('total_amount', spark_round(col('price') * col('quantity'), 2))
    
    print(f'Чистих рядків: {df_clean.count():,}')
    df_clean.filter(col('price').isNull()).count() == 0 and print('Nulls: 0 ✅')

---
## 3. Трансформація — типовий DE-звіт

Групуємо, агрегуємо, додаємо дати.

In [ ]:
if SPARK_OK:
    df_report = (
        df_clean
        .filter(col('status') == 'completed')
        .groupBy('category', 'product')
        .agg(
            count('*').alias('orders_count'),
            spark_sum('quantity').alias('total_units_sold'),
            spark_round(spark_sum('total_amount'), 2).alias('revenue'),
            spark_round(avg('price'), 2).alias('avg_price'),
        )
        .filter(col('orders_count') >= 100)
        .orderBy(col('revenue').desc())
    )
    
    print(f'Top products by revenue (min 100 orders): {df_report.count()}')
    df_report.show(10, truncate=False)

In [ ]:
if SPARK_OK:
    # Додаємо дати: рік-місяць для подальшої партиції
    df_final = (
        df_clean
        .withColumn('year', year(col('order_date')))
        .withColumn('month', month(col('order_date')))
        .withColumn('week_day', dayofweek(col('order_date')))
        .withColumn('is_weekend', when(col('week_day').isin(1, 7), lit(True)).otherwise(lit(False)))
    )
    
    print('Фінальний набір:')
    df_final.select('order_id', 'customer_id', 'product', 'total_amount', 'year', 'month', 'is_weekend').show(5)
    print(f'Років: {df_final.select("year").distinct().orderBy("year").rdd.flatMap(lambda x: x).collect()}')

---
## 4. Spark SQL

Реєструємо і пишемо SQL (як у продакшні).

In [ ]:
if SPARK_OK:
    df_final.createOrReplaceTempView('sales')
    
    result_sql = spark.sql('''
        SELECT 
            year,
            month,
            category,
            COUNT(DISTINCT customer_id) AS unique_customers,
            COUNT(*) AS orders,
            ROUND(SUM(total_amount), 2) AS revenue,
            ROUND(AVG(total_amount), 2) AS avg_order_value
        FROM sales
        WHERE status = 'completed'
        GROUP BY year, month, category
        HAVING orders > 50
        ORDER BY year, month, revenue DESC
    ''')
    
    print(f'SQL result: {result_sql.count():,} rows')
    result_sql.show(15, truncate=False)

---
## 5. Зберігаємо в Parquet (як у продакшні)

Партиційований запис — стандарт DE.

In [ ]:
if SPARK_OK:
    import os
    os.makedirs('demo_data', exist_ok=True)
    
    # Записуємо з партиціями по року та місяцю
    output_path = 'demo_data/sales_dwh'
    
    df_final.write \
        .mode('overwrite') \
        .partitionBy('year', 'month') \
        .parquet(output_path)
    
    print(f'Записано в {output_path}')
    
    # Перевіряємо
    df_check = spark.read.parquet(output_path)
    print(f'Прочитано назад: {df_check.count():,} rows')
    print(f'Партиції: {df_check.select("year", "month").distinct().count()}')

In [ ]:
if SPARK_OK:
    print('=== Partition layout ===')
    import subprocess
    result = subprocess.run(['cmd', '/c', 'dir', '/b', '/s', output_path], capture_output=True, text=True)
    lines = [l for l in result.stdout.split('\n') if 'year=' in l or '.parquet' in l.lower()]
    for l in lines[:15]:
        print(f'  {l.strip()}')
    if len(lines) > 15:
        print(f'  ... and {len(lines)-15} more')

---
## 6. Window Functions (віконні функції)

Рейтинг продуктів усередині категорії.

In [ ]:
if SPARK_OK:
    window_spec = Window.partitionBy('category').orderBy(col('revenue').desc())
    
    df_ranked = df_report.withColumn('rank', row_number().over(window_spec))
    
    print('=== Top-3 per category ===')
    df_ranked.filter(col('rank') <= 3).orderBy('category', 'rank').show(20, truncate=False)

---
## 7. Практичні завдання для самостійної роботи

1. Порахуйте **відсоток cancelled замовлень** по кожному payment_method
2. Знайдіть **топ-5 клієнтів** за revenue
3. Додайте колонку `price_category` (low < 50, medium 50-500, high > 500)
4. Порахуйте **кількість замовлень по днях тижня** (який день найпопулярніший?)
5. Збережіть результат п.4 в окремий Parquet файл
6. (Бонус) Зробіть **інкрементальне завантаження** — прочитайте тільки `year=2024`

In [ ]:
if SPARK_OK:
    # === Розв'язання ===
    
    # 1. Відсоток cancelled по payment_method
    print('=== 1. Cancelled by payment ===')
    spark.sql('''
        SELECT 
            payment_method,
            COUNT(*) AS total,
            SUM(CASE WHEN status = 'cancelled' THEN 1 ELSE 0 END) AS cancelled,
            ROUND(SUM(CASE WHEN status = 'cancelled' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS cancel_pct
        FROM sales
        GROUP BY payment_method
        ORDER BY cancel_pct DESC
    ''').show()
    
    # 2. Топ-5 клієнтів
    print('=== 2. Top-5 customers ===')
    top_customers = df_final.filter(col('status') == 'completed') \
        .groupBy('customer_id') \
        .agg(spark_round(spark_sum('total_amount'), 2).alias('total_revenue')) \
        .orderBy(col('total_revenue').desc()) \
        .limit(5)
    top_customers.show(truncate=False)

In [ ]:
if SPARK_OK:
    # 3. price_category
    df_cat = df_final.withColumn('price_category',
        when(col('price') < 50, 'low')
        .when(col('price') <= 500, 'medium')
        .otherwise('high')
    )
    df_cat.groupBy('price_category').count().show()
    
    # 4. По днях тижня
    print('=== 4. Orders by weekday ===')
    spark.sql('''
        SELECT 
            week_day,
            CASE week_day 
                WHEN 1 THEN 'Sunday' WHEN 2 THEN 'Monday' 
                WHEN 3 THEN 'Tuesday' WHEN 4 THEN 'Wednesday' 
                WHEN 5 THEN 'Thursday' WHEN 6 THEN 'Friday' 
                WHEN 7 THEN 'Saturday' 
            END AS day_name,
            COUNT(*) AS orders,
            ROUND(AVG(total_amount), 2) AS avg_revenue
        FROM sales
        GROUP BY week_day
        ORDER BY week_day
    ''').show()
    
    # 5. Збереження
    df_cat.select('week_day', 'order_id', 'total_amount').write \
        .mode('overwrite').parquet('demo_data/weekday_analysis')
    print('Збережено в demo_data/weekday_analysis')

In [ ]:
if SPARK_OK:
    spark.stop()
    print('Spark зупинено')

---
## Що ви навчились робити

| Задача | Як у DE |
|--------|---------|
| Перевірка якості даних | `isNull()`, `dropDuplicates()`, `approxQuantile()` |
| Заповнення null | `fillna()` з медіаною |
| Партиційований запис | `partitionBy('year', 'month').parquet()` |
| Агрегація | `groupBy().agg(sum, avg, count)` |
| Віконні функції | `Window.partitionBy().orderBy()` |
| Spark SQL | `createOrReplaceTempView()` + SQL |
| Інкрементальне читання | `spark.read.parquet('.../year=2024/')` |